# 3. Nyx on real data - rodents (mouse)

We use the Oxford Mouse Polysomnography Benchmark's *pilot*
set: a 12 h EEG/EMG recording from a freely behaving mouse during the light
phase, scored independently by ten experienced sleep researchers. We compare
against their consensus.

## How to use this notebook

It runs **one step per cell, and stops wherever there is something to decide**:

> ### STOP
> what to look at, and what to change if it is wrong

At each stop, look at the plot, change the setting in the cell below it if you
need to, and re-run from there. 
Once a recording is set up and you are happy, `score_recording` does all
of it in one line, and `nyx.open_queue` walks a whole manifest, one recording at a time.

> Cite the dataset if you use it: Brodersen et al., *The Oxford Mouse
> Polysomnography Benchmark Data Set*, Zenodo,
> [10.5281/zenodo.10200482](https://doi.org/10.5281/zenodo.10200482). CC BY 4.0.

## 1. Setup

In [ ]:
import os

for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_var] = "1"

import io
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nyx
from nyx import plotting

## 2. Download the data

78 MB into `examples/data/`, which is git-ignored. Skipped if it is already there.

In [ ]:
WHICH = "pilot"
RECORD = "10200482"
DATA = "data"

target = os.path.join(DATA, WHICH)
if os.path.isdir(target):
    print(f"already downloaded: {target}")
else:
    url = f"https://zenodo.org/api/records/{RECORD}/files/{WHICH}.zip/content"
    os.makedirs(DATA, exist_ok=True)
    print(f"downloading {WHICH}.zip ...")
    with urllib.request.urlopen(url, timeout=900) as response:
        payload = response.read()
    print(f"  {len(payload) / 1e6:.0f} MB, extracting ...")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        archive.extractall(DATA)
    print("done")

## 3. Load it

| | channel |
|---|---|
| frontal EEG | `Signal 0` |
| parietal EEG | `Signal 1` |
| **EMG** | **`Signal 3`** |

In [ ]:
recording = nyx.read_recording(
    f"{target}/recordings/{WHICH}.edf",
    eeg_channel="Signal 1",     # parietal EEG
    emg_channel="Signal 3",     # EMG
    name=WHICH,
)
params = nyx.load_params("mouse")

consensus = nyx.read_annotations(f"{target}/annotations/{WHICH}_consensus.hyp")

print(recording.describe())

totals = {}
for duration, label in zip(consensus["duration"], consensus["label"]):
    totals[str(label)] = totals.get(str(label), 0.0) + float(duration)
print("\nconsensus of 10 scorers:")
for label, seconds in sorted(totals.items()):
    print(f"  {label:<12} {seconds / 3600:5.2f} h  ({100 * seconds / 43200:.1f}%)")

> ### STOP — check channels
>
> Check the two channel names printed above against the table.
>
> The consensus vocabulary here is `awake` / `non-REM` / `REM` / `undefined`.
> nyx normalises common spellings when comparing, so `awake` matches `WAKE`
> without you mapping anything.

## 4. Look at the signal

Traces and spectrograms for the start and the end of the recording — the two
places artefacts live — plus a measurement of mains interference.

In [ ]:
check = nyx.check_signals(recording)
print(check.summary())
check.plot()
plt.show()

> ### STOP — mains, and where to start and stop
>
> **Mains.** Read the line-noise scores. Anything above 3 dB is worth notching,
> and you should be able to *see* it as a horizontal line in the spectrograms.
> Look especially at the **EMG** score: the EMG band is 30–100 Hz, so 50 or
> 60 Hz interference sits in the middle of the band the wake/sleep threshold is
> computed from, and can swamp it.
>
> **The window.** Look for flat, saturated or wildly noisy stretches at either
> end and set `WINDOW` inside them.
>
> Change either in the cell below and re-run from there.

In [ ]:
# A notch goes in the *params*, so it is recorded in run.json and replays.
NOTCH = check.suggested_notch()      # None, 50 or 60
print("notch:", NOTCH)

# params["EEG"]["notch"] = NOTCH
params["EMG"]["notch"] = NOTCH

# None = the whole recording, otherwise (start, end) in seconds.
WINDOW = (0, 42800)

In [ ]:
# Preprocessing is lazy, so this costs nothing until a trace is read.
processed = nyx.preprocess_recording(recording, params)
windowed = processed if WINDOW is None else processed.time_slice(*WINDOW)
window = (0.0, processed.duration) if WINDOW is None else WINDOW
print(f"analysing {window[0]:g}-{window[1]:g}s of {processed.duration:.0f}s")

In [ ]:
# Confirm the notch did what you wanted, on the signal that will actually be scored.
print(nyx.check_signals(windowed).summary())

## 5. EMG band power

The EMG spectrogram collapsed to one number per epoch: high in wake, low in
sleep.

In [ ]:
emg = nyx.compute_emg_features(windowed, params)

NOSIGNAL = 0.0     # power at or below this is signal loss, not sleep. 0 disables.
auto_threshold = nyx.find_wake_sleep_threshold(emg, nosignal_threshold=NOSIGNAL)

nyx.plot_emg_check(emg, threshold=auto_threshold, nosignal=NOSIGNAL)
plt.show()

print(f"automatic threshold {auto_threshold:.3f}   "
      f"-> {100 * (emg.power > auto_threshold).mean():.1f}% of epochs would be WAKE")

> ### STOP — the EMG threshold
>
> Two things to check:
>
> 1. **Does the histogram have two/three modes, with the cut in the valley between
>    them?** If it is one smooth blob, the EMG is not separating wake from sleep
>    and no threshold will fix that.
> 2. **Is the resulting proportion plausible?** This is a light-phase mouse
>    recording, so expect roughly 30% wake. If it says 99%, something is wrong —
>    go back and check the mains, and check you have the right channel.
>
> **If the EMG will not separate**, do not force it. Set the threshold *above*
> everything so every epoch starts as SLEEP, and let clustering recover wake
> from the EEG instead:
>
> ```python
> EMG_THRESHOLD = 1.1     # above the max of the min-max scaled power
> ```
>
> Then in section 7 use three clusters with `use_emg=True` and
> `STAGE_ORDER = ["REM", "NREM", "WAKE"]` — this is what
> `mouse_weak_emg` params does.

In [ ]:
# None = keep the automatic threshold. A number overrides it.
EMG_THRESHOLD = 0.55
nyx.plot_emg_check(emg, threshold=EMG_THRESHOLD, nosignal=NOSIGNAL)
plt.show()

**Or drag it.** Typing a number, re-running and looking is a slow way to
find a cut. `ThresholdSelector` puts the same threshold on both panels at
once and re-scores as you let go, so you can watch the wake percentage
move. It needs an interactive backend — `%matplotlib qt` locally, or
`%matplotlib widget` over SSH:

```python
%matplotlib qt
from nyx.interactive import ThresholdSelector

fig = nyx.plot_emg_check(emg, threshold=auto_threshold, nosignal=NOSIGNAL)
cut = ThresholdSelector(
    auto_threshold, hist_ax=fig.axes[0], trace_ax=fig.axes[1],
    on_release=lambda v: print(
        f"{v:.3f} -> {100 * (emg.power > v).mean():.1f}% wake"),
)
plt.show()

EMG_THRESHOLD = cut.value     # once you are happy with it
```

Drag either line, nudge with the arrow keys, or press `r` to go back to
where you started. The same class handles the no-signal cut — a second
`ThresholdSelector` on `fig.axes[0]` alone.

In [ ]:
%matplotlib widget
from nyx.interactive import ThresholdSelector

fig = nyx.plot_emg_check(emg, threshold=auto_threshold, nosignal=NOSIGNAL)
cut = ThresholdSelector(
    auto_threshold, hist_ax=fig.axes[0], trace_ax=fig.axes[1],
    on_release=lambda v: print(
        f"{v:.3f} -> {100 * (emg.power > v).mean():.1f}% wake"),
)
plt.show()



In [ ]:
EMG_THRESHOLD = cut.value 

In [ ]:
wake_sleep = nyx.classify_wake_sleep(
    emg,
    threshold=EMG_THRESHOLD,
    nosignal_threshold=NOSIGNAL,
    min_duration=params.get("scoring", {}).get("min_duration", 4.0),
)
print(f"using {wake_sleep.threshold:.3f} ({wake_sleep.threshold_source})")

durations = {}
for duration, label in zip(wake_sleep.hypnogram["duration"],
                           wake_sleep.hypnogram["label"]):
    durations[str(label)] = durations.get(str(label), 0.0) + float(duration)
total = sum(durations.values())
for label, seconds in sorted(durations.items()):
    print(f"  {label:<10} {seconds / 3600:5.2f} h  ({100 * seconds / total:.1f}%)")

nyx.plot_wake_sleep(wake_sleep)
plt.show()

## 6. EEG spectral PCA

PCA is fitted **on the sleep epochs only**.

In [ ]:
pca = nyx.compute_sleep_pca(windowed, params, wake_sleep)
print(f"{pca.scores.shape[0]} sleep epochs, {pca.scores.shape[1]} components")

nyx.plot_pca_grid(pca)
plt.show()

> ### STOP — do the components describe sleep?
>
> PC1 is usually a delta-vs-theta contrast — positive at low frequencies and
> negative around 6–9 Hz, or the other way round. **The sign is arbitrary**.
>
> If the first component explains more than 90% of the variance the signal has most likely a strong source of noise. Go back and check the signal again

## 7. Cluster the sleep epochs

In [ ]:
# Overrides DEFAULT_CLUSTERING and the params' "clustering" section.
clustering = {
    "method": "hdbscan",          # hdbscan | kmeans | gmm | elliptic
    "n_clusters": 2,              # kmeans / gmm / elliptic only
    "pcs_to_use": [0, 1, 2],
    "use_emg": False,             # add EMG power as an extra clustering feature
    "outlier_z_threshold": 6,     # beyond this |z| an epoch becomes NOSIGNAL
    "hdbscan_min_cluster_size": 300,
    "hdbscan_min_samples": 30,
}

clusters = nyx.cluster_sleep(pca, wake_sleep, clustering=clustering, emg=emg)
print("cluster sizes:", clusters.sizes())
if clusters.outlier_mask.any():
    print(f"outliers -> NOSIGNAL: {clusters.outlier_mask.sum()} "
          f"({100 * clusters.outlier_mask.mean():.1f}%)")

In [ ]:
# Scatter, coloured by each of the other features the clustering used (PC3, PC4, EMG power ...)
nyx.plot_cluster_features(clusters)
plt.show()

In [ ]:
nyx.plot_cluster_check(pca, clusters=clusters)
plt.show()

# The exact centroids, in the order stage_order names them.
nyx.cluster_ordering(clusters)

> ### STOP — which cluster is REM?
>
> **Read the right-hand panel, not the left one.** Clusters can look beautifully
> separated in PC space and still be assigned to the wrong stages.
>
> **NREM carries more low-frequency (delta, 1–4 Hz) power than REM.** REM has a
> theta peak around 6–9 Hz and much less delta. Note which cluster is which.
>
> Then read the **`order` column** of the table. `stage_order` in the next
> section names clusters by that position, not by their id — so `#0` gets the
> first name whatever it is called. When two centroids sit close together on PC1
> you cannot tell their order apart in the plot, and getting it wrong silently
> swaps two stages.
>
> If you got more clusters than you have stage names, either raise
> `hdbscan_min_cluster_size` or switch to `kmeans`/`gmm` with a fixed
> `n_clusters` or merge more clusters into one stage. Any cluster left unnamed is scored as NOSIGNAL, and nyx warns.

## 8. Name the clusters

`STAGE_ORDER` names clusters by their centroid on the first PC, lowest first.
The sign of a principal component is arbitrary, so **this order is a guess until
you have checked it against the spectra above.** If it is wrong, either flip
`STAGE_ORDER` or set `CLUSTER_OVERRIDES` by cluster id.

In [ ]:
STAGE_ORDER = ["REM", "NREM"]     # low PC1 -> high PC1
CLUSTER_OVERRIDES = None          # e.g. {0: "NREM", 1: "REM"}

staging = nyx.assign_stages(clusters, pca, wake_sleep,
                            stage_order=STAGE_ORDER, overrides=CLUSTER_OVERRIDES)

for cid, stage in sorted(staging.cluster_to_stage.items()):
    print(f"  cluster {cid}  PC1 centroid {clusters.centers[cid, 0]:+.3f}  ->  {stage}")

durations = staging.stage_durations()
total = sum(durations.values())
print()
for stage, seconds in sorted(durations.items()):
    print(f"  {stage:<10} {seconds / 3600:5.2f} h  ({100 * seconds / total:.1f}%)")

> ### STOP — are the proportions plausible?
>
> Compare against the consensus in section 3. In a light-phase mouse recording
> REM is a small fraction — a few percent of the total, and well under a third of
> NREM. If REM is coming out larger than NREM, the assignment is the wrong way
> round: flip `STAGE_ORDER` and re-run the cell above.

## 9. The result

In [ ]:
from nyx.pipeline import ScoringResult

agreement = nyx.evaluate(staging, consensus, window=window, verbose=False)

result = ScoringResult(
    recording=windowed, emg=emg, wake_sleep=wake_sleep, pca=pca,
    clusters=clusters, staging=staging, agreement=agreement,
    window=window, total_duration=recording.duration, params=params,
    reference=nyx.metrics.trim_manual_scores(consensus, *window),
)

print(agreement.summary())

In [ ]:
nyx.plot_scoring_overview(result)
plt.show()

## 10. Save

In [ ]:
OUTPUT_DIR = f"../results/{WHICH}"
nyx.save_results(result, OUTPUT_DIR)
print("saved to", os.path.abspath(OUTPUT_DIR))

`run.json` records the parameters and every decision you made above — the
window, the notch, the thresholds, the clustering settings and the cluster
mapping. Loading it replays this scoring exactly, with nothing left to choose:

```python
config = nyx.load_config(f"{OUTPUT_DIR}/run.json")
```

---

## Once it is set up

Everything above collapses into one call, which is what you use for rerunning recordings or testing those exact parameters:

```python
result = nyx.score_recording(recording, params, window=WINDOW,
                             emg_threshold=EMG_THRESHOLD,
                             stage_order=STAGE_ORDER,
                             cluster_overrides=CLUSTER_OVERRIDES,
                             reference=consensus)
```

For a whole study, put the recordings in a manifest and use the queue —
`nyx.open_queue`. It hands them out one at a time and remembers where you
got to, so you can work through a list without editing a path.